# Multi-Agent Systems — Supervisor + Specialists

## What You'll Learn
- The **Supervisor pattern** for multi-agent orchestration: one "manager" agent routes work to specialist agents
- How the `MultiAgentSupervisor` in AgentExplorr coordinates a **Researcher**, **Analyst**, and **Writer**
- Agent-to-agent communication via shared state and message passing
- When to use multi-agent systems vs. a single agent

## Prerequisites
```bash
pip install agentexplorr[agents]
# or: uv sync --extra agents
# Requires Ollama running locally: ollama pull llama3.2 && ollama serve
```

## Learning Resources
- [LangGraph Multi-Agent Tutorial](https://langchain-ai.github.io/langgraph/tutorials/multi_agent/multi-agent-collaboration/)
- [LangGraph Supervisor Pattern](https://langchain-ai.github.io/langgraph/tutorials/multi_agent/agent_supervisor/)
- [Paper: AutoGen — Enabling Next-Gen LLM Applications](https://arxiv.org/abs/2308.08155)
- [Paper: Communicative Agents for Software Dev (ChatDev)](https://arxiv.org/abs/2307.07924)
- [Video: Multi-Agent Systems with LangGraph](https://www.youtube.com/watch?v=hvAPnpSfSGo)
- [Video: AI Agent Teams (Supervisor Pattern)](https://www.youtube.com/watch?v=DjC6F-ByRHk)

In [ ]:
# Step 1: Explore the multi-agent architecture
#
# The MultiAgentSupervisor coordinates three specialist agents.
# Let's inspect its structure by importing the key components.

from agentexplorr.agents.multi_agent import (
    SPECIALISTS,
    SUPERVISOR,
    RESEARCHER,
    ANALYST,
    WRITER,
    SUPERVISOR_PROMPT,
    RESEARCHER_TOOLS,
    ANALYST_TOOLS,
    WRITER_TOOLS,
)

print("Multi-Agent System — Supervisor + Specialists")
print("=" * 55)
print()

# Show the specialist roster and their assigned tools
roster = {
    RESEARCHER: RESEARCHER_TOOLS,
    ANALYST:    ANALYST_TOOLS,
    WRITER:     WRITER_TOOLS,
}

print(f"Supervisor: '{SUPERVISOR}' — routes tasks to specialists")
print(f"Specialists: {SPECIALISTS}")
print()

for name, tools in roster.items():
    tool_names = [t.name for t in tools] if tools else ["(none — works from context)"]
    print(f"  {name.upper():12s} tools: {', '.join(tool_names)}")

print()
print("Architecture:")
print("""
       User Query
           |
           v
    +--------------+
    |  SUPERVISOR   |  <-- reads query, decides who works next
    +---+---+---+--+
        |   |   |
        v   v   v
   RES  ANA  WRI    <-- each specialist does its job
        |   |   |
        +---+---+
            |
            v
    +--------------+
    |  SUPERVISOR   |  <-- reviews output, routes again or FINISH
    +--------------+
""")

## Supervisor Pattern vs. Peer-to-Peer

There are two main architectures for multi-agent systems:

### Supervisor Pattern (used in AgentExplorr)
A central **supervisor** agent reads the user query and **routes** it to whichever specialist is needed. After each specialist finishes, control returns to the supervisor, which decides the next step or finishes.

**Advantages:**
- Clear chain of command — easy to understand and debug
- The supervisor maintains a global view of progress
- Specialists can be improved independently (focused prompts, focused tools)
- Natural ordering: research first, then analysis, then writing

**How routing works in AgentExplorr:**
The supervisor LLM responds with `ROUTE: researcher`, `ROUTE: analyst`, `ROUTE: writer`, or `ROUTE: FINISH`. The `_parse_routing()` function extracts the target from this text. If parsing fails, keyword-based fallback logic picks the most likely specialist. A safety valve limits the system to 8 routing rounds maximum.

### Peer-to-Peer Pattern (alternative)
Agents communicate directly with each other — no central coordinator. Each agent decides who to talk to next. Used by frameworks like **AutoGen** and **ChatDev**.

**Advantages:**
- More flexible for open-ended conversations
- No single point of failure (the supervisor)
- Can model real-world team dynamics

**Disadvantages:**
- Harder to debug (who talked to whom?)
- Risk of circular conversations (A calls B calls A calls B...)
- Requires each agent to know about all other agents

In [ ]:
# Step 2: Agent-to-agent communication via shared state
#
# In AgentExplorr's multi-agent system, agents don't send messages
# directly to each other. Instead, they communicate through SHARED STATE:
#
#   - MultiAgentState.messages       — full conversation history visible to all
#   - MultiAgentState.specialist_outputs — dict mapping agent name -> output text
#   - MultiAgentState.next_agent     — supervisor's routing decision
#   - MultiAgentState.iteration      — loop counter (safety limit)
#
# Let's model this communication pattern with a simple simulation.

from dataclasses import dataclass, field


@dataclass
class AgentMessage:
    """A message passed between agents via shared state."""
    sender: str
    content: str
    iteration: int


@dataclass
class SharedState:
    """Simplified model of MultiAgentState for demonstration."""
    messages: list[AgentMessage] = field(default_factory=list)
    specialist_outputs: dict[str, str] = field(default_factory=dict)
    next_agent: str = ""
    iteration: int = 0

    def add_message(self, sender: str, content: str) -> None:
        self.messages.append(AgentMessage(sender, content, self.iteration))

    def route_to(self, agent: str) -> None:
        self.next_agent = agent
        self.iteration += 1


# Simulate a typical multi-agent execution flow
state = SharedState()

# 1. User submits query
state.add_message("user", "Compare the populations of Tokyo and Paris")

# 2. Supervisor routes to researcher
state.route_to("researcher")
state.add_message("supervisor", "ROUTE: researcher — find population data")

# 3. Researcher produces findings
research_output = "Tokyo: ~14 million (city proper). Paris: ~2.1 million (city proper)."
state.specialist_outputs["researcher"] = research_output
state.add_message("researcher", research_output)

# 4. Supervisor routes to analyst
state.route_to("analyst")
state.add_message("supervisor", "ROUTE: analyst — compare the numbers")

# 5. Analyst produces analysis
analysis_output = "Tokyo is ~6.7x larger than Paris by city-proper population (14M vs 2.1M)."
state.specialist_outputs["analyst"] = analysis_output
state.add_message("analyst", analysis_output)

# 6. Supervisor routes to writer
state.route_to("writer")
state.add_message("supervisor", "ROUTE: writer — draft final response")

# 7. Writer synthesizes
writer_output = "Tokyo's city-proper population (~14M) is approximately 6.7 times that of Paris (~2.1M)."
state.specialist_outputs["writer"] = writer_output
state.add_message("writer", writer_output)

# 8. Supervisor finishes
state.route_to("FINISH")
state.add_message("supervisor", "ROUTE: FINISH — answer is complete")

# Display the execution trace
print("Simulated Multi-Agent Execution Trace")
print("=" * 55)
for msg in state.messages:
    tag = f"[{msg.sender.upper()}]"
    print(f"  iter {msg.iteration}  {tag:14s} {msg.content}")

print(f"\nTotal iterations: {state.iteration}")
print(f"Specialists used: {list(state.specialist_outputs.keys())}")

## When to Use Multi-Agent vs. Single Agent

Choosing between a single agent (ReAct or ToolAgent) and a multi-agent system depends on the **complexity and breadth** of the task.

### Use a Single Agent when:
- The task needs **one skill** (e.g., "calculate 2+3", "search for X")
- Latency matters — fewer LLM calls means faster responses
- The context window is sufficient for the entire task
- Debugging needs to be straightforward

### Use Multi-Agent when:
- The task requires **multiple distinct skills** (research + analysis + writing)
- You want **specialized prompts** — a single prompt that tries to cover everything performs worse than three focused prompts
- The task is **too complex for one reasoning chain** — long chains lose coherence
- You need **modularity** — swap out or upgrade specialists independently
- You want **least-privilege tool access** — the writer does not need search tools; the researcher does not need the calculator

### Real-world examples

| Task | Best approach | Why |
|------|--------------|-----|
| "What is 25 * 48?" | ToolAgent | Single tool call, no coordination needed |
| "Search for LangGraph" | ToolAgent | One search query, one answer |
| "Compare GDP of US, China, and Japan and write a report" | MultiAgent | Research (search data), Analysis (compare numbers), Writing (draft report) |
| "Analyze this dataset and create a presentation" | MultiAgent | Analysis specialist + writing specialist |

### Design principle: Least-privilege tools
In AgentExplorr's multi-agent system, each specialist only gets the tools it needs:
- **Researcher** gets `web_search` and `web_scrape` (but NOT the calculator)
- **Analyst** gets `calculator` (but NOT search tools)
- **Writer** gets no tools at all (it works purely from the shared context)

This prevents mistakes (the writer accidentally searching when it should be writing) and reduces confusion in the LLM's decision-making.

## Key Takeaways

1. **Multi-agent systems use division of labor** — instead of one monolithic agent, specialized agents handle research, analysis, and writing independently with focused prompts and tools.
2. **The Supervisor pattern provides structured coordination** — one central agent reads the task, routes to specialists, reviews outputs, and decides when to finish. This is easier to debug than peer-to-peer.
3. **Shared state enables agent communication** — agents do not talk directly to each other. Instead, they read from and write to a shared `MultiAgentState` (messages + specialist_outputs). Each specialist sees what came before.
4. **Least-privilege tool access reduces errors** — giving each specialist only the tools it needs prevents the LLM from making confused decisions (e.g., the writer searching instead of writing).
5. **Safety valves prevent runaway loops** — the system limits the supervisor to 8 routing rounds and each specialist to 5 tool-calling rounds. If parsing the routing decision fails, fallback logic and a `FINISH` default ensure the system terminates.

## Try It Yourself
```python
# Run the full MultiAgentSupervisor (requires Ollama with llama3.2):
#   ollama pull llama3.2 && ollama serve
from agentexplorr.agents.multi_agent import MultiAgentSupervisor
system = MultiAgentSupervisor(verbose=True)
result = system.run(
    "Compare the populations of Tokyo and New York City, "
    "and analyze which is growing faster."
)
print(result.final_answer)
print(f"Specialists used: {list(result.specialist_outputs.keys())}")
print(f"Routing rounds: {result.iterations}")
```

## Next Steps
- Read the source: `src/agentexplorr/agents/multi_agent.py`
- Review the [ReAct Agent notebook](01_react_agent.ipynb) for comparison with single-agent patterns
- Review the [Tool-Using Agent notebook](02_tool_using_agent.ipynb) to understand the tools these specialists use
- Explore related frameworks: [CrewAI](https://github.com/joaomdmoura/crewAI), [AutoGen](https://github.com/microsoft/autogen), [ChatDev](https://github.com/OpenBMB/ChatDev)